# Supply Chain Shipping Delay Prediction — Decision Tree Regression

## Business Problem

The goal is to predict the **number of shipping delay days** for a future trade-route operation using operational and environmental factors.

### Target
`shipping_delay_days`

### Main question
> Based on weather, port congestion, container availability, geopolitical risk, route characteristics, freight conditions, and shipping method, how many days is a route operation likely to be delayed?

This notebook applies a **Decision Tree Regressor** to the same processed dataset and the same core features used in the Linear Regression notebook.

We will learn:

1. How a Decision Tree Regressor works
2. Why scaling is not required for a tree
3. How to train an unrestricted tree
4. How to detect overfitting
5. How to tune tree hyperparameters
6. How pruning/regularization works in a tree
7. How to evaluate MAE, RMSE, and R²
8. How to visualize the tree
9. How to interpret feature importance
10. How to explain the results from a business perspective


## 1. Import Libraries


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 2. Load the Processed Dataset

We use the dataset created after data cleaning, integration, and feature engineering in the EDA notebook.


In [ ]:
def find_processed_file():
    possible_paths = [
        Path.cwd() / "data" / "processed" / "supply_chain_ml_ready.csv",
        Path.cwd().parent / "data" / "processed" / "supply_chain_ml_ready.csv",
        Path("/mnt/data/data/processed/supply_chain_ml_ready.csv"),
    ]

    for path in possible_paths:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find supply_chain_ml_ready.csv. "
        "Please run the EDA notebook first."
    )


DATA_PATH = find_processed_file()

print("Using dataset:", DATA_PATH.resolve())

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"],
)

# Sort by time because we want older data for training
# and newer/future data for testing.
df = df.sort_values("date").reset_index(drop=True)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Remove Redundant / Target-Leakage Columns

We keep `container_availability_index` and remove `container_shortage_score` because both contain the same information in opposite directions.

We also do **not** use columns such as `delay_category` or `route_status` because they can directly reveal the delay outcome.

The target itself (`shipping_delay_days`) must never be included in X.


In [ ]:
TARGET = "shipping_delay_days"

# Remove the redundant feature if it is present.
df.drop(
    columns=["container_shortage_score"],
    inplace=True,
    errors="ignore",
)

print("Target:", TARGET)
print(
    "container_shortage_score present:",
    "container_shortage_score" in df.columns,
)


## 4. Select the Input Features

We use the same core predictors used in the earlier Linear Regression work.

### Numeric features
- Trade volume
- Freight cost per tonne
- Container availability
- Port congestion
- Fuel cost
- Commodity stress
- Weather disruption
- Geopolitical risk
- Distance
- Estimated transit days
- Month

### Categorical features
- Shipping method
- Trade-route type


In [ ]:
numeric_features = [
    "trade_volume_tonnes",
    "freight_cost_per_tonne",
    "container_availability_index",
    "port_congestion_index",
    "fuel_cost_index",
    "commodity_stress_index",
    "weather_disruption_score",
    "geopolitical_risk_score",
    "distance_km",
    "estimated_transit_days",
    "month",
]

categorical_features = [
    "shipping_method",
    "trade_route_type",
]

model_features = numeric_features + categorical_features

X = df[model_features].copy()
y = df[TARGET].copy()

print("Number of original input features:", len(model_features))
print("Target:", TARGET)

display(X.head())


## 5. Time-Based Train/Test Split

We use the first **80%** of the observations for training and the last **20%** for testing.

Why?

The business problem is to predict future delays, so a chronological split is more realistic than randomly mixing old and future observations.


In [ ]:
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

train_dates = df["date"].iloc[:split_index]
test_dates = df["date"].iloc[split_index:]

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows : {len(X_test):,}")

print(
    f"Training period: "
    f"{train_dates.min().date()} to {train_dates.max().date()}"
)
print(
    f"Testing period : "
    f"{test_dates.min().date()} to {test_dates.max().date()}"
)


## 6. Preprocessing for Decision Tree

A Decision Tree **does not require StandardScaler**.

Why?

A tree makes decisions such as:

`port_congestion_index <= 58.4`

It compares feature values with thresholds. Scaling does not change the ordering of the values, so standardization is unnecessary.

However, categorical variables still need to be converted to numbers. We use **OneHotEncoder** for them.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        # Keep numeric values as they are.
        ("num", "passthrough", numeric_features),

        # Convert categories such as Air, Sea, Road, Rail
        # into numeric dummy columns.
        (
            "cat",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
    ],
    verbose_feature_names_out=False,
)


# 7. First Decision Tree — No Restrictions

We first allow the tree to grow without restricting its depth.

This is useful because it demonstrates one of the biggest weaknesses of Decision Trees:

> **A very deep tree can memorize the training data and overfit.**


In [ ]:
unrestricted_tree = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                random_state=42,
            ),
        ),
    ]
)

unrestricted_tree.fit(X_train, y_train)

train_pred_unrestricted = unrestricted_tree.predict(X_train)
test_pred_unrestricted = unrestricted_tree.predict(X_test)


## 8. Evaluation Function

For regression we evaluate:

- **MAE** — average absolute prediction error in days
- **RMSE** — gives larger prediction mistakes more penalty
- **R²** — how much variation in shipping delay is explained by the model


In [ ]:
def regression_metrics(y_actual, y_predicted):
    mae = mean_absolute_error(y_actual, y_predicted)
    rmse = np.sqrt(
        mean_squared_error(y_actual, y_predicted)
    )
    r2 = r2_score(y_actual, y_predicted)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }


unrestricted_train_metrics = regression_metrics(
    y_train,
    train_pred_unrestricted,
)

unrestricted_test_metrics = regression_metrics(
    y_test,
    test_pred_unrestricted,
)

comparison_unrestricted = pd.DataFrame(
    [
        {
            "Dataset": "Training",
            **unrestricted_train_metrics,
        },
        {
            "Dataset": "Testing",
            **unrestricted_test_metrics,
        },
    ]
)

display(comparison_unrestricted)


## 9. What Does the Unrestricted Tree Tell Us?

If the training results are almost perfect but the test results are much worse, the model is **overfitting**.

An unrestricted Decision Tree can continue splitting until individual training records are isolated.

That produces:

- Very low training error
- Very high training R²
- Much weaker test performance

This is a **high-variance model**.


In [ ]:
r2_gap_unrestricted = (
    unrestricted_train_metrics["R2"]
    - unrestricted_test_metrics["R2"]
)

print(
    "Train-Test R² gap:",
    round(r2_gap_unrestricted, 4),
)

print(
    "Unrestricted tree depth:",
    unrestricted_tree
    .named_steps["model"]
    .get_depth(),
)

print(
    "Unrestricted number of leaves:",
    unrestricted_tree
    .named_steps["model"]
    .get_n_leaves(),
)


# 10. Decision Tree Hyperparameters

To control overfitting, we restrict the tree using hyperparameters.

### `max_depth`
Maximum number of levels in the tree.

- Small depth → simpler tree
- Very large depth → more complex tree and possible overfitting

### `min_samples_split`
Minimum number of records required before a node is allowed to split.

Higher value → fewer unnecessary splits.

### `min_samples_leaf`
Minimum number of observations that must remain in a leaf.

Higher value → more stable predictions.

### `ccp_alpha`
Cost-complexity pruning parameter.

Higher `ccp_alpha` → more branches are removed → simpler tree.

These are the Decision Tree version of controlling **model complexity**.


# 11. Hyperparameter Tuning

We use `GridSearchCV` with `TimeSeriesSplit`.

The test set is **not used** to select the hyperparameters.

Grid Search tries different combinations and selects the combination with the best cross-validation RMSE.


In [ ]:
time_cv = TimeSeriesSplit(n_splits=5)

tree_for_tuning = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                random_state=42,
            ),
        ),
    ]
)

# A focused grid keeps the notebook practical while still testing
# simple vs more complex tree settings.
param_grid = {
    "model__max_depth": [3, 5, 7, 10],
    "model__min_samples_split": [10, 50],
    "model__min_samples_leaf": [10, 50],
    "model__ccp_alpha": [0.0, 0.001, 0.005],
}

grid_search = GridSearchCV(
    estimator=tree_for_tuning,
    param_grid=param_grid,
    cv=time_cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print()
print(
    "Best cross-validation RMSE:",
    round(-grid_search.best_score_, 4),
)


# 12. Tuned Decision Tree

Now we evaluate the best tree selected using the training-period cross-validation.


In [ ]:
best_tree = grid_search.best_estimator_

train_pred_tuned = best_tree.predict(X_train)
test_pred_tuned = best_tree.predict(X_test)

tuned_train_metrics = regression_metrics(
    y_train,
    train_pred_tuned,
)

tuned_test_metrics = regression_metrics(
    y_test,
    test_pred_tuned,
)

tuned_results = pd.DataFrame(
    [
        {
            "Dataset": "Training",
            **tuned_train_metrics,
        },
        {
            "Dataset": "Testing",
            **tuned_test_metrics,
        },
    ]
)

display(tuned_results)

print(
    "Tuned tree depth:",
    best_tree.named_steps["model"].get_depth(),
)

print(
    "Tuned tree leaves:",
    best_tree.named_steps["model"].get_n_leaves(),
)


# 13. Untuned vs Tuned Tree

This comparison demonstrates why controlling tree complexity matters.


In [ ]:
tree_comparison = pd.DataFrame({
    "Model": [
        "Unrestricted Decision Tree",
        "Tuned Decision Tree",
    ],
    "Train_MAE": [
        unrestricted_train_metrics["MAE"],
        tuned_train_metrics["MAE"],
    ],
    "Test_MAE": [
        unrestricted_test_metrics["MAE"],
        tuned_test_metrics["MAE"],
    ],
    "Train_RMSE": [
        unrestricted_train_metrics["RMSE"],
        tuned_train_metrics["RMSE"],
    ],
    "Test_RMSE": [
        unrestricted_test_metrics["RMSE"],
        tuned_test_metrics["RMSE"],
    ],
    "Train_R2": [
        unrestricted_train_metrics["R2"],
        tuned_train_metrics["R2"],
    ],
    "Test_R2": [
        unrestricted_test_metrics["R2"],
        tuned_test_metrics["R2"],
    ],
})

tree_comparison["R2_Gap"] = (
    tree_comparison["Train_R2"]
    - tree_comparison["Test_R2"]
)

display(tree_comparison)


### Interpretation

The unrestricted tree normally has extremely strong training performance because it memorizes the training observations.

The tuned tree deliberately accepts some training error in exchange for better performance on future/test data.

This is the **bias-variance trade-off**:

- Unrestricted tree → low bias, very high variance
- Tuned tree → slightly higher bias, lower variance, better generalization


# 14. Actual vs Predicted Shipping Delay

If the model were perfect, every point would fall exactly on the diagonal line.


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    test_pred_tuned,
    alpha=0.25,
    s=15,
)

low = min(
    y_test.min(),
    test_pred_tuned.min(),
)

high = max(
    y_test.max(),
    test_pred_tuned.max(),
)

plt.plot(
    [low, high],
    [low, high],
    linestyle="--",
)

plt.xlabel("Actual Shipping Delay (Days)")
plt.ylabel("Predicted Shipping Delay (Days)")
plt.title("Decision Tree: Actual vs Predicted Shipping Delay")
plt.tight_layout()
plt.show()


## 15. Residual Analysis

Residual:

`Actual Delay - Predicted Delay`

- Positive residual → model underpredicted the delay
- Negative residual → model overpredicted the delay
- Residual close to 0 → prediction was close to actual


In [ ]:
test_residuals = y_test - test_pred_tuned

plt.figure(figsize=(8, 5))

plt.scatter(
    test_pred_tuned,
    test_residuals,
    alpha=0.25,
    s=15,
)

plt.axhline(
    0,
    linestyle="--",
)

plt.xlabel("Predicted Shipping Delay (Days)")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Decision Tree Residual Plot")
plt.tight_layout()
plt.show()

print(
    "Average residual:",
    round(test_residuals.mean(), 4),
)


# 16. Visualize the Decision Tree

A complete tree may contain many nodes, so we display only the first few levels.

Each node contains a rule such as:

`weather_disruption_score <= value`

The model follows the left or right branch until it reaches a leaf.  
The leaf contains the predicted average shipping delay.


In [ ]:
feature_names = (
    best_tree
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

tree_model = best_tree.named_steps["model"]

plt.figure(figsize=(24, 12))

plot_tree(
    tree_model,
    feature_names=feature_names,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
)

plt.title("Tuned Decision Tree — First 3 Levels")
plt.show()


# 17. Feature Importance

Decision Trees calculate feature importance based on how much each feature reduces prediction error when it is used for splitting.

A larger importance means the feature played a larger role in the tree's predictions.

Important:

> Feature importance shows predictive usefulness in this model. It does not prove that the feature causes shipping delays.


In [ ]:
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": tree_model.feature_importances_,
})

feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(feature_importance.head(20))


In [ ]:
top_features = (
    feature_importance
    .head(15)
    .sort_values("Importance")
)

plt.figure(figsize=(9, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"],
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top Decision Tree Features for Shipping Delay")
plt.tight_layout()
plt.show()


# 18. Understand One Prediction

This section takes one future/test record and shows:

- Actual shipping delay
- Predicted shipping delay
- Prediction error
- Input values used by the model

This makes the model result easier to explain in business terms.


In [ ]:
example_position = 0

example_X = X_test.iloc[[example_position]]
example_actual = y_test.iloc[example_position]
example_prediction = best_tree.predict(example_X)[0]

print(
    "Actual shipping delay:",
    round(example_actual, 2),
    "days",
)

print(
    "Predicted shipping delay:",
    round(example_prediction, 2),
    "days",
)

print(
    "Absolute prediction error:",
    round(
        abs(example_actual - example_prediction),
        2,
    ),
    "days",
)

print()
print("Input values:")
display(example_X.T.rename(columns={example_X.index[0]: "Value"}))


# 19. Business Interpretation

A Decision Tree prediction can be explained using a sequence of rules.

Conceptually, the tree may learn patterns such as:

```text
Port congestion high?
        |
       Yes
        |
Weather disruption high?
        |
       Yes
        |
Container availability low?
        |
       Yes
        |
Predicted shipping delay = higher
```

The exact rules come from the fitted tree, not from manually written business rules.

### Business value

A logistics team can use the prediction to:

- identify route operations likely to experience longer delays
- investigate high-risk operating conditions
- allocate additional buffer time
- warn customers earlier
- consider alternate routing or transportation plans


# 20. Decision Tree vs Linear Regression

### Linear Regression

Learns one global linear relationship:

`Delay = b0 + b1x1 + b2x2 + ...`

It works well when relationships are approximately linear.

### Decision Tree

Learns a sequence of threshold-based rules.

It can automatically capture:

- nonlinear relationships
- interactions between features
- different behavior in different parts of the data

Example:

Weather may matter much more when port congestion is already high.  
A Decision Tree can learn this interaction without manually creating an interaction term.


# 21. Key Decision Tree Concepts From This Project

### Root Node
The first split in the tree.

### Internal Node
A decision/rule inside the tree.

### Leaf Node
The final node containing the predicted shipping-delay value.

### Split
A rule such as:

`port_congestion_index <= 55`

### Tree Depth
Number of levels in the tree.

### Overfitting
Tree becomes too deep and memorizes training data.

### Pruning / Regularization
Reduce tree complexity using:

- `max_depth`
- `min_samples_split`
- `min_samples_leaf`
- `ccp_alpha`

### Feature Importance
Shows which features were most useful for reducing prediction error in the tree.


# 22. Final Model Summary


In [ ]:
final_summary = pd.DataFrame({
    "Item": [
        "Algorithm",
        "Target",
        "Train MAE",
        "Test MAE",
        "Train RMSE",
        "Test RMSE",
        "Train R²",
        "Test R²",
        "Tree Depth",
        "Number of Leaves",
    ],
    "Value": [
        "Tuned Decision Tree Regressor",
        TARGET,
        round(tuned_train_metrics["MAE"], 4),
        round(tuned_test_metrics["MAE"], 4),
        round(tuned_train_metrics["RMSE"], 4),
        round(tuned_test_metrics["RMSE"], 4),
        round(tuned_train_metrics["R2"], 4),
        round(tuned_test_metrics["R2"], 4),
        tree_model.get_depth(),
        tree_model.get_n_leaves(),
    ],
})

display(final_summary)


# 23. Final Conclusion

A simple way to explain this project section:

> I applied a Decision Tree Regressor to predict `shipping_delay_days`. First, I trained an unrestricted tree and compared its training and test performance. The unrestricted model heavily overfit the training data, which demonstrated the high-variance behavior of Decision Trees. I then used time-series cross-validation and GridSearchCV to tune `max_depth`, `min_samples_split`, `min_samples_leaf`, and `ccp_alpha`. The tuned tree reduced overfitting and generalized much better to the future test period. I evaluated the final model using MAE, RMSE, and R², visualized its decision rules, and used feature importance to identify the operational factors that contributed most to shipping-delay predictions.

## Main lesson

> **Decision Trees are powerful because they learn nonlinear rules and feature interactions, but controlling tree depth and complexity is essential to prevent overfitting.**
